In [10]:
# === INFERENCE NOTEBOOK ===
import xgboost as xgb
import pandas as pd
import numpy as np
import pickle

MODEL_PATH     = "models/baseline_xgboost.json"
ARTIFACTS_PATH = "models/preprocessing_info.pkl"
CSV_PATH       = "data/flows.csv"   # <-- The input 80 feature network flow data



In [11]:
# 1. Load model and artifacts
model = xgb.XGBClassifier()
model.load_model(MODEL_PATH)

with open(ARTIFACTS_PATH, "rb") as f:
    artifacts = pickle.load(f)

le_target    = artifacts["target_encoder"]      # was "label_encoder"
feature_cols = artifacts["feature_columns"]     # was "feature_cols"

Dictionary successfully converted to a pickle file.


In [12]:
with open("rename_map.pkl", "rb") as file:
    rename_map = pickle.load(file)
# print(type(rename_map),rename_map)

In [14]:
# 2. Load and preprocess input CSV
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)


df.rename(columns=rename_map, inplace=True)

# CICIDS2017 has a duplicate 'Fwd Header Length' column recorded as 'Fwd Header Length.1'
# The model expects it — just copy the value from the renamed column
df['Fwd Header Length.1'] = df['Fwd Header Length']

# Verify all expected features are now present
missing = [c for c in feature_cols if c not in df.columns]
print(f"Still missing after rename: {missing if missing else 'None — all good!'}")

Still missing after rename: None — all good!


In [15]:
# 2. Load and preprocess input CSV
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Add our rename label mapping to the df
df.rename(columns=rename_map, inplace=True)
df['Fwd Header Length.1'] = df['Fwd Header Length']

df.dropna(subset=feature_cols, inplace=True)
X = df[feature_cols]

In [16]:
# 3. Run inference
y_pred       = model.predict(X)
y_pred_proba = model.predict_proba(X)
labels       = le_target.inverse_transform(y_pred)


In [14]:
# 4. Output results
results = df[feature_cols].copy()
results["Predicted_Label"]    = labels
results["Confidence"]         = y_pred_proba.max(axis=1).round(4)
results["Predicted_Class_ID"] = y_pred

print(results[["Predicted_Label", "Confidence"]].value_counts())
print(results[["Predicted_Label", "Confidence"]].head(20))

results.to_csv("inference_results.csv", index=False)

Predicted_Label  Confidence
BENIGN           0.9999        194
                 0.9996         49
                 0.9998         37
                 0.9990         19
                 1.0000          7
                 0.9994          1
                 0.9989          1
                 0.9986          1
                 0.9995          1
                 0.9949          1
Name: count, dtype: int64
   Predicted_Label  Confidence
0           BENIGN      0.9999
1           BENIGN      0.9998
2           BENIGN      0.9998
3           BENIGN      0.9999
4           BENIGN      0.9996
5           BENIGN      0.9996
6           BENIGN      0.9998
7           BENIGN      0.9999
8           BENIGN      0.9999
9           BENIGN      0.9999
10          BENIGN      0.9998
11          BENIGN      0.9999
12          BENIGN      0.9998
13          BENIGN      0.9998
14          BENIGN      0.9998
15          BENIGN      0.9999
16          BENIGN      0.9998
17          BENIGN      0.9999
18     

Still missing after rename: None — all good!


In [8]:
import pickle
with open("models/preprocessing_info.pkl", "rb") as f:
    artifacts = pickle.load(f)

print(type(artifacts))
print(artifacts.keys() if hasattr(artifacts, 'keys') else dir(artifacts))

<class 'dict'>
dict_keys(['categorical_encoders', 'target_encoder', 'feature_columns', 'categorical_columns'])


In [20]:
import pandas as pd
df = pd.read_csv("data/flows.csv")
df.columns = df.columns.str.strip()
print(df.columns.tolist())
print(f"\nShape: {df.shape}")

['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s', 'bwd_pkts_s', 'tot_fwd_pkts', 'tot_bwd_pkts', 'totlen_fwd_pkts', 'totlen_bwd_pkts', 'fwd_pkt_len_max', 'fwd_pkt_len_min', 'fwd_pkt_len_mean', 'fwd_pkt_len_std', 'bwd_pkt_len_max', 'bwd_pkt_len_min', 'bwd_pkt_len_mean', 'bwd_pkt_len_std', 'pkt_len_max', 'pkt_len_min', 'pkt_len_mean', 'pkt_len_std', 'pkt_len_var', 'fwd_header_len', 'bwd_header_len', 'fwd_seg_size_min', 'fwd_act_data_pkts', 'flow_iat_mean', 'flow_iat_max', 'flow_iat_min', 'flow_iat_std', 'fwd_iat_tot', 'fwd_iat_max', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_tot', 'bwd_iat_max', 'bwd_iat_min', 'bwd_iat_mean', 'bwd_iat_std', 'fwd_psh_flags', 'bwd_psh_flags', 'fwd_urg_flags', 'bwd_urg_flags', 'fin_flag_cnt', 'syn_flag_cnt', 'rst_flag_cnt', 'psh_flag_cnt', 'ack_flag_cnt', 'urg_flag_cnt', 'ece_flag_cnt', 'down_up_ratio', 'pkt_size_avg', 'init_fwd_win_byts', 'init_bwd_win_byts', 'activ